In [ ]:
# ===================================================================================
# IMAGE SERVICE MIGRATION SCRIPT (CLONE PORTAL ITEM + URL REWRITE)
# - Image Services are server-registered; the raster data lives on the filesystem.
# - This script clones the portal item and rewrites the URL to the target server.
# - Image data must be registered on the target ArcGIS Server separately.
# ===================================================================================

import pandas as pd
import json
import copy
import time
import csv
import os
import datetime
import urllib3
import requests
import sys
import shutil
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from arcgis.gis import GIS

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# =============================================================================
# --- CONFIGURATION (from shared config) ---------------------------------------
# =============================================================================
from migration_config import *

# --- ORCHESTRATOR SIDECAR LOADER ---
_sidecar_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "_sidecar_11_image_services.json")
if os.path.exists(_sidecar_path):
    import json as _json
    IMAGE_SERVICE_IDS = _json.load(open(_sidecar_path))["ids"]
    print(f"[Orchestrator] Loaded {len(IMAGE_SERVICE_IDS)} Image Service IDs from sidecar.")
else:
    IMAGE_SERVICE_IDS = [
        # Example Source IDs
        # "abc123def456...",
    ]

# =============================================================================
# --- CONNECT ------------------------------------------------------------------
# =============================================================================
if not os.path.exists(TEMP_DIR): os.makedirs(TEMP_DIR)

print("Connecting...")
try:
    session = requests.Session()
    retry_strategy = Retry(total=5, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    source_gis = GIS(url=SOURCE_URL, token=SOURCE_TOKEN, verify_cert=False, referer=SOURCE_URL, session=session)
    target_gis = GIS(url=TARGET_URL, token=TARGET_TOKEN, verify_cert=False, referer=TARGET_URL, session=session)

    print(f"   > Source Connected: {source_gis.url}")
    if not target_gis.users.me: raise Exception("Target login failed.")
    print(f"   > Target Connected: {target_gis.users.me.username}")

except Exception as e:
    print(f"\n❌ CRITICAL CONNECTION FAILURE: {e}")
    sys.exit(1)

# =============================================================================
# --- LOGGING ------------------------------------------------------------------
# =============================================================================
STATS = {"scanned": 0, "cloned": 0, "skipped_log": 0, "failures": 0}

ALREADY_MIGRATED_IDS = set()
if os.path.exists(LOG_FILE):
    try:
        df_log = pd.read_csv(LOG_FILE)
        if "SourceID" in df_log.columns:
            ALREADY_MIGRATED_IDS = set(df_log["SourceID"].astype(str).str.strip())
        print(f"Loaded history: {len(ALREADY_MIGRATED_IDS)} items previously migrated.")
    except: pass
else:
    try:
        with open(LOG_FILE, mode="w", newline="") as f:
            csv.writer(f).writerow(["SourceID", "LayerIndex", "TargetID", "Title", "MigratedDate", "Type"])
    except: pass

def log_migration(source_id, index, target_id, title, item_type):
    try:
        with open(LOG_FILE, mode="a", newline="") as f:
            csv.writer(f).writerow([
                source_id, index, target_id, title,
                datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"), item_type
            ])
            ALREADY_MIGRATED_IDS.add(str(source_id))
    except: pass

# =============================================================================
# --- HELPERS ------------------------------------------------------------------
# =============================================================================
def get_source_folder_name(source_item):
    try:
        if not source_item.ownerFolder: return None
        user = source_gis.users.get(source_item.owner)
        if user:
            for f in user.folders:
                fid = f['id'] if isinstance(f, dict) else f.id
                ftitle = f['title'] if isinstance(f, dict) else f.title
                if fid == source_item.ownerFolder: return ftitle
    except: pass
    return None

def ensure_target_folder_exists(username, folder_name):
    try:
        user = target_gis.users.get(username)
        if not user: return False
        existing_folders = []
        for f in user.folders:
            if hasattr(f, 'title'):
                existing_folders.append(f.title)
            elif isinstance(f, dict):
                existing_folders.append(f.get('title'))
        if folder_name in existing_folders: return True
        print(f"      + Creating folder '{folder_name}' for user '{username}'...")
        target_gis.content.create_folder(folder_name, owner=username)
        return True
    except Exception as e:
        print(f"      ⚠ Folder creation error: {e}")
        return False

def assign_correct_owner_and_folder(source_item, target_item):
    try:
        source_owner = source_item.owner
        target_owner_to_use = DEFAULT_OWNER
        target_folder_to_use = DEFAULT_FOLDER

        found_user = target_gis.users.get(source_owner)

        if found_user:
            print(f"      👤 Source owner '{source_owner}' found in target.")
            target_owner_to_use = source_owner
            src_folder_name = get_source_folder_name(source_item)
            if src_folder_name:
                if ensure_target_folder_exists(target_owner_to_use, src_folder_name):
                    target_folder_to_use = src_folder_name
                else:
                    print(f"      ⚠️ Could not create folder '{src_folder_name}'. Using Default.")
            else:
                target_folder_to_use = None
        else:
            print(f"      👤 Source owner '{source_owner}' NOT found. Defaulting to '{DEFAULT_OWNER}'.")
            ensure_target_folder_exists(DEFAULT_OWNER, DEFAULT_FOLDER)

        print(f"      📂 Moving to: Owner={target_owner_to_use}, Folder={target_folder_to_use}")
        target_item.reassign_to(target_owner_to_use, target_folder=target_folder_to_use)

    except Exception as e:
        print(f"      ⚠️ Reassign/Move Failed: {e} (Item remains with Creator)")

def copy_thumbnail(source_item, target_item):
    try:
        print("      🖼️ Copying Thumbnail...")
        temp_dir = "thumbnails_temp"
        os.makedirs(temp_dir, exist_ok=True)
        thumb_path = source_item.download_thumbnail(save_folder=temp_dir)
        if thumb_path: target_item.update(thumbnail=thumb_path)
    except: pass

def mirror_source_sharing(source_item, target_item):
    try:
        print("      👥 Mirroring Sharing Permissions (Source -> Target)...")

        is_public = False
        is_org = False

        if source_item.access == 'public':
            is_public = True
            is_org = True
        elif source_item.access == 'org':
            is_org = True

        source_groups = []
        try:
            raw_groups = source_item.sharing.groups
            if isinstance(raw_groups, list):
                source_groups = raw_groups
            else:
                raise ValueError("Not a list")
        except:
            raw_shared = source_item.shared_with
            if isinstance(raw_shared, dict) and 'groups' in raw_shared:
                source_groups = raw_shared['groups']
            elif isinstance(raw_shared, list):
                source_groups = raw_shared

        target_group_ids = []

        if source_groups:
            print(f"         - Found {len(source_groups)} source groups. Mapping...")
            for sg in source_groups:
                sg_title = sg.title if hasattr(sg, 'title') else str(sg)
                found_groups = target_gis.groups.search(f"title:\"{sg_title}\"")
                matched_group = next((g for g in found_groups if g.title == sg_title), None)
                if matched_group:
                    target_group_ids.append(matched_group.id)
                    print(f"           + Mapped: '{sg_title}'")
                else:
                    print(f"           - Skipped: '{sg_title}' (Not found in Target)")

        target_item.share(everyone=is_public, org=is_org, groups=target_group_ids)
        print(f"         ✅ Shared (Org={is_org}, Public={is_public}, Groups={len(target_group_ids)})")

    except Exception as e:
        print(f"      ⚠ Sharing Mirror Failed: {e}")

# =============================================================================
# --- MAIN LOOP ----------------------------------------------------------------
# =============================================================================
print(f"\nStarting Image Service Migration...")
start_time = time.time()

# Extract hostnames for URL rewriting
source_host = SOURCE_URL.split("//")[1].split("/")[0]
target_host = TARGET_URL.split("//")[1].split("/")[0]
print(f"   URL Rewrite: {source_host} -> {target_host}")

for source_id in IMAGE_SERVICE_IDS:
    try:
        STATS["scanned"] += 1

        # CHECK: HISTORY LOG
        if str(source_id) in ALREADY_MIGRATED_IDS:
            print(f"\n[Skip] {source_id} found in log.")
            STATS["skipped_log"] += 1
            continue

        source_item = source_gis.content.get(source_id)
        if not source_item:
            print(f"❌ Source {source_id} not found.")
            STATS["failures"] += 1
            continue

        print(f"\nProcessing: {source_item.title} ({source_id})...")

        # --- DETERMINE TARGET TITLE ---
        target_title = source_item.title
        if APPEND_MIGRATED:
            target_title = f"{source_item.title} (Migrated)"

        # --- STEP 1: BUILD TAGS ---
        final_tags = list(source_item.tags or [])
        if "Migrated" not in final_tags: final_tags.append("Migrated")
        tag_marker = f"SourceID:{source_id}"
        if tag_marker not in final_tags: final_tags.append(tag_marker)

        # --- STEP 2: REWRITE URL ---
        rewritten_url = ""
        if source_item.url:
            rewritten_url = source_item.url.replace(source_host, target_host)
            print(f"      URL: {source_item.url}")
            print(f"       -> {rewritten_url}")

        # --- STEP 3: CLONE ITEM ON TARGET ---
        print(" > 1. Creating Image Service item on target...")
        item_props = {
            "type": source_item.type,
            "title": target_title,
            "url": rewritten_url,
            "snippet": source_item.snippet,
            "description": source_item.description,
            "tags": final_tags,
            "typeKeywords": list(source_item.typeKeywords) if source_item.typeKeywords else [],
            "accessInformation": source_item.accessInformation,
            "licenseInfo": source_item.licenseInfo,
        }

        new_item = target_gis.content.add(item_props, folder=DEFAULT_FOLDER)
        if not new_item:
            raise Exception("Failed to create item on target.")
        print(f"      ✅ Created: {new_item.id} ({new_item.title})")

        # --- STEP 4: THUMBNAIL ---
        copy_thumbnail(source_item, new_item)

        # --- STEP 5: SHARING (BEFORE owner reassignment) ---
        mirror_source_sharing(source_item, new_item)

        # --- STEP 6: OWNER & FOLDER ---
        print(" > 2. Assigning Owner & Folder...")
        assign_correct_owner_and_folder(source_item, new_item)

        # --- STEP 7: LOG ---
        log_migration(source_id, "N/A", new_item.id, new_item.title, "Image Service")
        STATS["cloned"] += 1
        print(f"🚀 SUCCESS: Cloned Image Service '{new_item.title}'")
        print(f"   ℹ️  Image data must be registered on the target server separately.")

        print(f"   💤 Cooling down for {THROTTLE_SECONDS} seconds...")
        time.sleep(THROTTLE_SECONDS)

    except Exception as e:
        print(f"❌ FAILED on {source_id}: {e}")
        STATS["failures"] += 1

# =============================================================================
# --- REPORT -------------------------------------------------------------------
# =============================================================================
end_time = time.time()
total_seconds = int(end_time - start_time)
duration_str = f"{total_seconds // 3600}h {(total_seconds % 3600) // 60}m {total_seconds % 60}s"

print("\n" + "=" * 60)
print("        IMAGE SERVICE MIGRATION REPORT        ")
print("=" * 60)
print(f" ⏱️  Total Duration:             {duration_str}")
print("-" * 60)
print(f" 📦 Scanned:                   {STATS['scanned']}")
print(f" 🧾 Skipped (log):             {STATS['skipped_log']}")
print(f" ✅ Cloned:                    {STATS['cloned']}")
print(f" ❌ Failures:                  {STATS['failures']}")
print("-" * 60)
print(f" ℹ️  NOTE: Image data must be registered on the target ArcGIS Server.")
print("=" * 60)